# Set 07 – Textmerkmale und Multinomial Naive Bayes

Modelle können Text nicht direkt verarbeiten. Zuerst wird jede Nachricht in numerische Merkmale umgewandelt.

Bag of Words verwendet Wörter oder Wortgruppen als Spalten und deren Häufigkeiten als Werte. Multinomial Naive Bayes eignet sich für solche nichtnegativen Zähldaten.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

## 1. Kleiner kontrollierter Nachrichtendatensatz

Die Nachrichten sind absichtlich einfach formuliert. Die spätere Übung nutzt einen separaten realen SMS-Datensatz.

In [ ]:
ham_nachrichten = [
    "Treffen wir uns morgen um zehn im Büro",
    "Kannst du bitte Milch und Brot kaufen",
    "Der Termin wurde auf Montag verschoben",
    "Vielen Dank für deine Hilfe gestern",
    "Ich rufe dich nach der Arbeit an",
    "Bitte sende mir die Präsentation",
    "Das Abendessen ist um sieben fertig",
    "Wir sehen uns später am Bahnhof",
    "Die Rechnung liegt im gemeinsamen Ordner",
    "Herzlichen Glückwunsch zum Geburtstag",
    "Kannst du den Bericht heute prüfen",
    "Der Zug kommt zehn Minuten später",
    "Unser Meeting beginnt um neun Uhr",
    "Ich habe die Tickets bereits gekauft",
    "Bitte bestätige den Arzttermin morgen",
    "Das Paket wurde heute geliefert",
    "Kommst du am Wochenende zum Grillen",
    "Die Schlüssel liegen auf dem Tisch",
    "Wir brauchen noch Kaffee für das Büro",
    "Danke für die schnelle Rückmeldung",
]

spam_nachrichten = [
    "Gewinne jetzt kostenlos einen Gutschein",
    "Exklusiver Preis wartet klicke sofort",
    "Du hast einen Jackpot gewonnen",
    "Kostenloses Angebot nur heute verfügbar",
    "Jetzt anrufen und Bargeld gewinnen",
    "Sichere dir deinen gratis Bonus",
    "Glückwunsch du bist der Gewinner",
    "Klicke hier für deinen Sofortgewinn",
    "Verdiene schnell Geld ohne Risiko",
    "Letzte Chance auf einen gratis Preis",
    "Dringend bestätige deinen Gewinn jetzt",
    "Kostenlos registrieren und Bonus erhalten",
    "Gewinne ein neues Smartphone sofort",
    "Dein exklusiver Gutschein ist bereit",
    "Nur heute hundert Prozent kostenlos",
    "Antworte jetzt und erhalte Bargeld",
    "Sonderangebot klicke auf diesen Link",
    "Du wurdest für einen Preis ausgewählt",
    "Gratis Gewinn wartet auf Bestätigung",
    "Jetzt handeln und Bonus sichern",
]

texte = ham_nachrichten + spam_nachrichten
labels = ["ham"] * len(ham_nachrichten) + ["spam"] * len(spam_nachrichten)
daten = pd.DataFrame({"text": texte, "label": labels}).sample(frac=1, random_state=42).reset_index(drop=True)
display(daten.head())
print(daten["label"].value_counts())

## 2. CountVectorizer Schritt für Schritt

fit() lernt das Vokabular aus den Texten. transform() zählt die Wörter in jeder Nachricht. fit_transform() erledigt beides direkt nacheinander.

Die entstehende Matrix ist meist dünn besetzt und wird deshalb als Sparse Matrix gespeichert.

In [ ]:
vectorizer_demo = CountVectorizer(lowercase=True)
matrix_demo = vectorizer_demo.fit_transform(daten["text"].head(6))
woerter_demo = vectorizer_demo.get_feature_names_out()

print("Matrixform:", matrix_demo.shape)
display(pd.DataFrame(
    matrix_demo.toarray(),
    columns=woerter_demo,
).iloc[:, :15])

## 3. Pipeline aus Wortzählung und MultinomialNB

Die Aufteilung erfolgt vor dem Lernen des Vokabulars. Dadurch gelangen Wörter aus Testnachrichten nicht in die Feature-Auswahl.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    daten["text"],
    daten["label"],
    test_size=0.25,
    random_state=42,
    stratify=daten["label"],
)

pipeline_alle_woerter = Pipeline([
    ("woerter", CountVectorizer(lowercase=True, ngram_range=(1, 2))),
    ("modell", MultinomialNB(alpha=1.0)),
])
pipeline_alle_woerter.fit(X_train, y_train)

pred_alle = pipeline_alle_woerter.predict(X_test)
print(classification_report(y_test, pred_alle, zero_division=0))

## 4. Welche Wörter sprechen für welche Klasse?

MultinomialNB speichert logarithmierte Wortwahrscheinlichkeiten pro Klasse. Die Differenz Log-P(spam) minus Log-P(ham) liefert ein einfaches Ranking.

Positive Werte sprechen im Modell eher für Spam, negative eher für Ham.

In [ ]:
vectorizer = pipeline_alle_woerter.named_steps["woerter"]
modell = pipeline_alle_woerter.named_steps["modell"]
feature_namen = vectorizer.get_feature_names_out()

spam_index = list(modell.classes_).index("spam")
ham_index = list(modell.classes_).index("ham")
log_verhaeltnis = modell.feature_log_prob_[spam_index] - modell.feature_log_prob_[ham_index]

wort_ranking = pd.DataFrame({
    "Merkmal": feature_namen,
    "Log_Verhaeltnis_Spam_zu_Ham": log_verhaeltnis,
}).sort_values("Log_Verhaeltnis_Spam_zu_Ham")

display(wort_ranking.head(12))
display(wort_ranking.tail(12).sort_values("Log_Verhaeltnis_Spam_zu_Ham", ascending=False))

In [ ]:
ham_top = wort_ranking.head(10)
spam_top = wort_ranking.tail(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(ham_top["Merkmal"], ham_top["Log_Verhaeltnis_Spam_zu_Ham"], color="#4C78A8")
axes[0].set_title("spricht eher für Ham")
axes[1].barh(spam_top["Merkmal"], spam_top["Log_Verhaeltnis_Spam_zu_Ham"], color="#E45756")
axes[1].set_title("spricht eher für Spam")
for ax in axes:
    ax.set_xlabel("Log-Wahrscheinlichkeitsverhältnis")
plt.tight_layout()
plt.show()

## 5. Automatische Merkmalsauswahl mit Chi²

SelectKBest bewertet, wie stark jedes Wortmerkmal mit der Klasse zusammenhängt. Nur die k besten Merkmale werden an das Modell weitergegeben.

Die Auswahl muss innerhalb der Pipeline liegen, damit sie ausschließlich aus Trainingsdaten gelernt wird.

In [ ]:
pipeline_auswahl = Pipeline([
    ("woerter", CountVectorizer(lowercase=True, ngram_range=(1, 2))),
    ("auswahl", SelectKBest(score_func=chi2, k=20)),
    ("modell", MultinomialNB(alpha=1.0)),
])
pipeline_auswahl.fit(X_train, y_train)

pred_auswahl = pipeline_auswahl.predict(X_test)
print(classification_report(y_test, pred_auswahl, zero_division=0))

## 6. Ausgewählte Wörter anzeigen

In [ ]:
vectorizer = pipeline_auswahl.named_steps["woerter"]
selector = pipeline_auswahl.named_steps["auswahl"]
alle_namen = vectorizer.get_feature_names_out()
ausgewaehlte_namen = alle_namen[selector.get_support()]

auswahl_tabelle = pd.DataFrame({
    "Merkmal": alle_namen,
    "Chi2_Score": selector.scores_,
    "ausgewählt": selector.get_support(),
}).sort_values("Chi2_Score", ascending=False)

display(auswahl_tabelle.head(25).round(3))
print("Ausgewählte Merkmale:")
print(ausgewaehlte_namen)

## 7. Neue Nachrichten klassifizieren

Die Pipeline wendet Vokabular, Auswahl und Modell automatisch in derselben Reihenfolge an.

In [ ]:
neue_nachrichten = [
    "Bitte sende mir den Bericht bis morgen",
    "Glückwunsch klicke jetzt für deinen gratis Gewinn",
    "Das Treffen beginnt heute um vier",
]
proba = pipeline_auswahl.predict_proba(neue_nachrichten)
ergebnis = pd.DataFrame(proba, columns=pipeline_auswahl.named_steps["modell"].classes_)
ergebnis.insert(0, "Nachricht", neue_nachrichten)
ergebnis["Vorhersage"] = pipeline_auswahl.predict(neue_nachrichten)
display(ergebnis.round(3))

## Wichtige Grenzen

- Wortreihenfolge geht bei reinen Unigrammen verloren.
- Unbekannte Wörter werden ignoriert.
- Naive Bayes behandelt Wortmerkmale bedingt unabhängig.
- Auswahl kann seltene, aber fachlich wichtige Wörter entfernen.
- Ein kleines kontrolliertes Korpus sagt wenig über reale Generalisierung aus.
- Reale Nachrichten verändern sich über Zeit; Spamfilter müssen überwacht werden.